<a href="https://colab.research.google.com/github/traviswheeler/CompBioAsia/blob/main/3-CompBioAsia_SecondaryStructure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting secondary structure

This is our final project: We are going to build a model that can predict secondary structure of protein sequences. 

#### Setup
Standard stuff - run each cell in this section once.

In [ ]:
# Imports
import numpy as np
import torch
from torch import nn
import gzip

# Download the CullPDB 5926 (filtered) secondary-structure dataset.
# Hosted as a release asset on this repo for a fast, stable download.
# (Original Princeton source is dead; this is the pi-helix-corrected
#  "updated" variant, same 57-feature layout.)
!wget -O data.npy.gz "https://github.com/traviswheeler/CompBioAsia/releases/download/data-v1/cullpdb_5926_filtered_updated.npy.gz"

# Load the data and reshape into (proteins, 700 positions, 57 features).
with gzip.open('data.npy.gz', 'rb') as f:
    x = np.load(f)

x = x.reshape(-1, 700, 57)
aminos = torch.transpose(torch.tensor(x[:,:,0:22]), -1, -2).float()
labels = torch.transpose(torch.tensor(x[:,:,22:31]), -1, -2).float()

aminos_train = aminos[:5000]
aminos_test = aminos[5000:]

labels_train = labels[:5000]
labels_test = labels[5000:]

print(aminos.shape, labels.shape)
print(aminos_train.shape, aminos_test.shape)
print(labels_train.shape, labels_test.shape)


alphabet = ['A', 'C', 'E', 'D', 'G', 'F', 'I', 'H', 'K', 'M', 'L', 'N', 'Q', 'P', 'S', 'R', 'T', 'W', 'V', 'Y', 'X','NoSeq']
alpha_to_num = {c:i for i, c in enumerate(alphabet)}
secondary_labels = ['L', 'B', 'E', 'G', 'I', 'H', 'S', 'T','NoSeq']
label_to_num = {c:i for i, c in enumerate(secondary_labels)}

#### Take a look at the data

Each protein is a chain of amino acids, and every amino acid has a secondary-structure label (the local 3-D shape it folds into: helix, strand, loop, ...). That per-residue label is what we're trying to predict.

**Re-run the cell below to see a different random protein each time.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# Full names of the 8 secondary-structure classes (there are 8, plus a 9th
# "NoSeq" padding token -- not 10; the colormap just happens to have 10 colors).
ss_names = {'L':'loop / irregular', 'B':'beta-bridge', 'E':'beta-strand',
            'G':'3-10 helix', 'I':'pi-helix', 'H':'alpha-helix',
            'S':'bend', 'T':'turn', 'NoSeq':'padding'}

# --- pick a random protein (re-run this cell to see a new one) ---
idx = np.random.randint(len(aminos))

aa_i = aminos[idx].argmax(0).numpy()        # amino-acid index at each position
ss_i = labels[idx].argmax(0).numpy()        # secondary-structure index at each position
L = int((aa_i != alphabet.index('NoSeq')).sum())   # real length (drop NoSeq padding)

seq = "".join(alphabet[i]         for i in aa_i[:L])
sst = "".join(secondary_labels[i] for i in ss_i[:L])

print(f"Protein #{idx}   (length {L} residues)\n")
for s in range(0, L, 60):
    print("seq ", seq[s:s+60])
    print("ss  ", sst[s:s+60])
    print()

# consistent colors across the strip, the bar, and the legend
colors = plt.cm.tab10(np.arange(len(secondary_labels)))   # one color per SS class

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 3.6),
                               gridspec_kw={'height_ratios': [1, 3]})
# colored strip: secondary structure along the chain
ax1.imshow(ss_i[:L][None, :], aspect='auto',
           cmap=ListedColormap(colors), vmin=0, vmax=len(secondary_labels) - 1)
ax1.set_yticks([]); ax1.set_xlabel("residue position")
ax1.set_title(f"secondary structure along protein #{idx}")

# bar: this protein's composition (8 real classes, drop NoSeq)
counts = np.bincount(ss_i[:L], minlength=len(secondary_labels))[:8]
ax2.bar(secondary_labels[:8], counts, color=colors[:8])
ax2.set_ylabel("count"); ax2.set_title("this protein's secondary-structure composition")

# legend mapping each label letter -> color + full name (the 8 real classes)
handles = [mpatches.Patch(color=colors[k], label=f"{secondary_labels[k]} - {ss_names[secondary_labels[k]]}")
           for k in range(8)]
# reserve space on the right so the legend sits INSIDE the figure (won't get clipped)
fig.subplots_adjust(left=0.07, right=0.78, top=0.90, bottom=0.16, hspace=0.7)
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(0.80, 0.5),
           title="secondary-structure labels", frameon=False)
plt.show()

In [ ]:
def train_with_data(x, y,
                    model, batch_size, 
                    steps, learning_rate, 
                    loss_function, checkin=100):
  dev = 'cpu'
  if torch.cuda.is_available():
    dev = 'cuda:0'

  model.to(dev)
  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

  for step in range(steps):
    batch_i = torch.randint(0, len(x), (batch_size,))
    batch_x = x[batch_i].to(dev)
    batch_y = y[batch_i].to(dev)

    out = model(batch_x)

    loss = loss_function(out, batch_y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if step % checkin == 0:
      print('progress: {:.2%} '.format(step / steps), 'loss:', loss.item())

  model.to('cpu')

# This function tests our accuracy.
# For each real residue (NoSeq padding ignored), the model's "call" is the class
# with the largest output (argmax). It's correct if that call matches the true
# label.
# accuracy = (# residues where the call is the correct class) / (# residues)
def test_accuracy(x, y, model, batch_size=32):
  dev = 'cpu'
  if torch.cuda.is_available():
    dev = 'cuda:0'

  model.to(dev)

  correct = 0
  total = 0
  with torch.no_grad():
    batches = int(len(x) / batch_size) - 1  # we lose a little bit of data but that's ok

    for batch in range(batches):
      start_idx = batch * batch_size

      batch_x = x[start_idx:start_idx+batch_size].to(dev)
      batch_y = y[start_idx:start_idx+batch_size].to(dev)

      out = model(batch_x)

      call = torch.argmax(out, dim=1)           # the model's pick at each residue
      true = torch.argmax(batch_y, dim=1)       # the correct class at each residue

      real = batch_y[:, -1, :] == 0             # ignore NoSeq padding positions
      correct += torch.sum((call == true) & real)
      total   += torch.sum(real)

  model.to('cpu')
  return float(correct / total)


#### Build, train, and test our model

The cell below builds a small convolutional network, measures its accuracy on the
test set **before** training, trains it, then measures accuracy **again**. The
cells after it then improve the model one step at a time.

**Accuracy** is per-residue. For each real residue (padding ignored), the model's
prediction is the class it scores highest (`argmax`), and we report the fraction
of residues it gets right. With 8 possible structure classes, an untrained
network sits around ~25-30%; training should push it higher.

**Loss** is the number training actually minimizes. We use `CrossEntropyLoss`,
the standard loss for picking one class out of many. For each residue the model
emits 9 raw scores (*logits*); cross-entropy runs them through `softmax` to get
class probabilities, then charges a penalty of `-log(probability it gave the true
class)`. Put high probability on the right class → small penalty; be confidently
wrong → large penalty. The loss is the average of that penalty over all real
residues. Because `softmax` is applied inside the loss, the model ends in a plain
`Conv1d` with **no** `Sigmoid`, and `ignore_index=8` drops the NoSeq padding
positions so they don't drown out the real residues.

**Reading the model definition.** Each `nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding)`
slides learnable filters along the sequence:

- `in_channels` (first layer = **22**): the size of the vector describing each
  residue coming *in* — here the 22-dim amino-acid encoding.
- `out_channels` (first layer = **10**): how many new features the layer produces
  per residue. The layer learns this many filters; each outputs one channel.
- `kernel_size` (first layer = **1**): how many neighboring residues each filter
  looks at. `1` means "this residue only" (no context); `5` means a window of 5.
- `stride=1`: move the window one residue at a time (don't skip positions).
- `padding='same'`: pad the ends so the output length matches the input (700),
  keeping the per-residue predictions lined up with the labels.

**Why two of them?** The first `Conv1d` turns the 22-dim amino-acid encoding into
10 learned features per residue; an `ELU` adds a nonlinearity; the second
`Conv1d` takes those 10 features and, over a 5-residue window, produces the 9
class scores. Stacking layers (with a nonlinearity between) lets the network build
richer features and see more sequence context than any single layer could. Note
the chain has to match up: the first layer's `out_channels` (10) is the second
layer's `in_channels` (10), and the last layer's `out_channels` must be 9 — one
score per class.

In [ ]:
batch_size = 32 # How many images we train on at every step
steps = 2000 # How many total steps we will train for
learning_rate = 0.0001 # How fast we adjust gradients with gradient descent
checkin = int(steps / 5) # How often we print our loss (smaller = more frequently)

# CrossEntropyLoss is the natural loss for "pick one class per residue". It
# applies softmax internally, so the model should output raw scores (logits) --
# note there is no Sigmoid on the end of the model below.
# It wants the true class as an index (0-8), and `ignore_index=8` drops the
# NoSeq padding positions from the loss. Our labels are stored one-hot, so this
# wrapper converts them to indices on the fly.
cross_entropy = nn.CrossEntropyLoss(ignore_index=8)  # 8 = NoSeq padding
def loss_function(out, target_onehot):
    return cross_entropy(out, torch.argmax(target_onehot, dim=1))


# Here's our starting place.
# To add more Conv1d layers you'll want to use:
# nn.Conv1d(input_channels, output_channels, kernel_size=your_kernel_size, stride=1, padding='same')
# You can replace input_channels, output_channels, and kernel_size with whatever you want

model = nn.Sequential(nn.Conv1d(22, 10, kernel_size=1, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(10, 9, kernel_size=5, stride=1, padding='same'))

print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))



train_with_data(aminos_train, labels_train, model, 
                batch_size, steps, learning_rate, 
                loss_function, checkin=checkin)


print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))

#### Step 1: give the first layer some context (kernel_size 1 → 5)

In the starting model the first `Conv1d` used `kernel_size=1`, so it looked at
each residue completely on its own — a per-residue feature transform with no
sense of neighbors. Only the final layer (kernel 5) saw any context at all.

Here we change just one thing: the first layer's `kernel_size` from `1` to `5`.
Now its filters span a 5-residue window, so the network starts picking up local
patterns from the very first layer instead of postponing all context to the end.
Run it and compare the trained accuracy to the starting model above.

In [ ]:
# Step 1: give the FIRST Conv1d a kernel_size of 5 (was 1).
# With kernel_size=1 the first layer saw each residue in isolation; widening it to
# 5 lets the very first features already depend on a 5-residue window, so the
# network starts learning local context instead of leaving all of it to the last
# layer. Everything else is unchanged.
# (Reuses batch_size, steps, learning_rate, checkin, loss_function.)
model = nn.Sequential(nn.Conv1d(22, 10, kernel_size=5, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(10, 9, kernel_size=5, stride=1, padding='same'))

print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))
train_with_data(aminos_train, labels_train, model,
                batch_size, steps, learning_rate,
                loss_function, checkin=checkin)
print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))

#### Step 2: widen the receptive field

A residue's **receptive field** is the stretch of the sequence that can influence
its prediction. One `kernel_size=5` conv gives each output a receptive field of 5
(±2 neighbors). But secondary-structure elements are longer than that — an
alpha-helix can run a dozen-plus residues — so a narrow window can't "see" the
whole pattern it's part of.

Two ways to widen it: a bigger kernel, or **stacking convolutions**. When you
stack them the receptive field grows: each output of the second conv already
summarizes a window of the first conv's outputs, which each summarize a window of
the input. Below we add a third layer with `kernel_size=11`; combined with the
surrounding width-5 layers, each prediction now draws on ~19 residues — enough to
span a whole secondary-structure element.

In [ ]:
# Step 2: add a third Conv1d layer with a wide kernel (11) to widen the
# receptive field. Stacking convs means each output residue now depends on a much
# longer stretch of sequence (~19 residues here), enough to span a whole helix or
# strand. Note the new layer keeps the channel chain intact: 22 -> 10 -> 10 -> 9.
# (Reuses batch_size, steps, learning_rate, checkin, loss_function.)
model = nn.Sequential(nn.Conv1d(22, 10, kernel_size=5, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(10, 10, kernel_size=11, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(10, 9, kernel_size=5, stride=1, padding='same'))

print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))
train_with_data(aminos_train, labels_train, model,
                batch_size, steps, learning_rate,
                loss_function, checkin=checkin)
print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))

#### Step 3: wider layers (more output channels)

So far each hidden layer produces only 10 features per residue. Bumping
`out_channels` to **32** gives every layer more room: instead of 10 detectors for
local amino-acid patterns, it can learn 32. More channels = more capacity to
represent different kinds of structural cues (and more parameters to train). It's
a cheap, effective knob — widening often helps more than you'd expect, right up
until the model starts to overfit (which we'll address next).

In [ ]:
# Step 3: widen the layers to 32 output channels.
# Each Conv1d now learns 32 feature detectors per residue instead of 10, giving
# the network a richer vocabulary of amino-acid "motifs" to describe local
# structure. Channel counts still have to chain: 22 -> 32 -> 32 -> 9.
# (Reuses batch_size, steps, learning_rate, checkin, loss_function.)
model = nn.Sequential(nn.Conv1d(22, 32, kernel_size=5, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(32, 32, kernel_size=11, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Conv1d(32, 9, kernel_size=5, stride=1, padding='same'))

print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))
train_with_data(aminos_train, labels_train, model,
                batch_size, steps, learning_rate,
                loss_function, checkin=checkin)
print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))

#### Step 4: regularize with dropout — then keep tinkering

As models get bigger they can start to **overfit**: memorizing quirks of the
training proteins instead of learning rules that generalize to new ones. A simple
guard is **dropout** — during each training step it randomly switches off a
fraction of the features (here 30%), forcing the network to spread what it learns
across many features rather than relying on a lucky few. It's only active while
training and turns off automatically at test time.

That's the toolkit. From here, **the model is yours to improve** — try combining
and pushing these knobs:

1. **Deeper** — add more `Conv1d` + `ELU` blocks.
2. **Wider** — more `out_channels` per layer.
3. **Bigger kernels / receptive field** — let each residue see more neighbors.
4. **Dropout rate** — more if you're overfitting, less if it's underfitting.
5. **Training** — try a larger `learning_rate` (e.g. `0.001`) and more `steps`.
6. **Other activations** — see https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity
7. **Other loss functions** — though `CrossEntropyLoss` is hard to beat here.

Watch both numbers: if training loss keeps dropping but test accuracy stalls or
falls, you're overfitting (regularize or shrink); if both stay low, you're
underfitting (go bigger or train longer).

In [ ]:
# Step 4: add Dropout for regularization.
# Dropout randomly zeros some of the features during each training step, so the
# network can't lean too hard on any single one -- it has to learn redundant,
# more robust patterns. This fights overfitting (memorizing the training set
# instead of learning general rules), which matters more as the model gets bigger.
# Dropout is only active during training; it turns itself off automatically at
# test time. (Reuses batch_size, steps, learning_rate, checkin, loss_function.)
model = nn.Sequential(nn.Conv1d(22, 32, kernel_size=5, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Dropout(0.3),
                      nn.Conv1d(32, 32, kernel_size=11, stride=1, padding='same'),
                      nn.ELU(),
                      nn.Dropout(0.3),
                      nn.Conv1d(32, 9, kernel_size=5, stride=1, padding='same'))

print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))
train_with_data(aminos_train, labels_train, model,
                batch_size, steps, learning_rate,
                loss_function, checkin=checkin)
print("Accuracy: ", round(test_accuracy(aminos_test, labels_test, model), 3))